[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/pandera-certified/notebooks/day-01-dataframe-schema-basics.ipynb#scrollTo=a1b2c3d4)

---
# Day 1 · DataFrameSchema Basics — Columns, Types, and Coercion
**certified-journeys / pandera-certified** · Day 1 · Schema Foundations

> **Goal for today:** Define a `DataFrameSchema`, validate a real Pandas DataFrame, handle nullable columns and type coercion, and understand exactly what a `SchemaError` tells you.


In [ ]:
%pip install -q pandera


## Step 1 · What is DataFrameSchema?

Pandera's `DataFrameSchema` is the dict-style API for declaring what a DataFrame
**must** look like: which columns exist, their dtypes, and any value-level checks.
Validation is eager — it runs the moment you call `.validate(df)`.

Key objects you'll use today:

| Object | Purpose |
|---|---|
| `pa.DataFrameSchema` | Top-level schema container |
| `pa.Column(dtype, ...)` | Declares one column's type and options |
| `pa.SchemaError` | Exception raised on validation failure |

Official quickstart: https://pandera.readthedocs.io/en/stable/dataframe_schemas.html


In [ ]:
import pandera as pa
import pandas as pd

# Define a schema for an employee table
schema = pa.DataFrameSchema({
    "name":   pa.Column(str),          # must be string dtype
    "age":    pa.Column(int),          # must be integer
    "salary": pa.Column(float),        # must be float
})

# Build a valid DataFrame
df = pd.DataFrame({
    "name":   ["Alice", "Bob", "Carol"],
    "age":    [30, 25, 40],
    "salary": [90000.0, 75000.0, 120000.0],
})

# Validate returns the same DataFrame if it passes
validated = schema.validate(df)
print(validated)
print("\nDtypes after validation:")
print(validated.dtypes)


### What just happened?

- **`schema.validate(df)` returns the DataFrame** — validation is non-destructive; you can chain it inline.
- Pandera checked that all three columns exist, none are missing, and the dtypes match.
- **No error = schema satisfied.** If any check had failed, a `SchemaError` would have been raised immediately.
- The schema acts like a contract: downstream code can trust the types without extra `isinstance` guards.


## Step 2 · Nullable columns — handling NaN safely

By default, Pandera **rejects** `NaN` / `None` in any column. Set `nullable=True`
on a `Column` to explicitly allow missing values. This is the right place to
document your data contract — if NaN is expected, say so.

```python
pa.Column(float, nullable=True)   # NaN allowed
pa.Column(float)                  # NaN rejected (default)
```


In [ ]:
import numpy as np

# Schema where salary is optional (employees may be contractors)
schema_nullable = pa.DataFrameSchema({
    "name":   pa.Column(str),
    "age":    pa.Column(int),
    "salary": pa.Column(float, nullable=True),  # allow missing salary
})

df_with_nan = pd.DataFrame({
    "name":   ["Alice", "Bob", "Carol"],
    "age":    [30, 25, 40],
    "salary": [90000.0, np.nan, 120000.0],   # Bob has no salary on record
})

validated = schema_nullable.validate(df_with_nan)
print("Validated with NaN:\n", validated)

# Now show what happens WITHOUT nullable=True
schema_strict = pa.DataFrameSchema({
    "name":   pa.Column(str),
    "age":    pa.Column(int),
    "salary": pa.Column(float),               # nullable=False (default)
})

try:
    schema_strict.validate(df_with_nan)
except pa.errors.SchemaError as e:
    print("\nExpected error:", e)


### What just happened?

- **`nullable=True` is an explicit contract:** it signals that downstream code must handle `NaN`.
- Without `nullable=True`, Pandera catches the `NaN` and raises `SchemaError` immediately — this is the desired safety net.
- **NaN in int columns is special:** Pandas uses `Int64` (nullable integer) or promotes to `float64`. Declare `pa.Column(float, nullable=True)` for int-like columns that may have gaps.
- Always prefer explicit `nullable=True/False` over leaving it implicit — it documents your intent.


## Step 3 · Type coercion — `coerce=True`

Real-world DataFrames often arrive from CSV or JSON with every column as `object`
(string). Pandera's `coerce=True` lets you **declare the target type and cast
automatically** rather than manually calling `.astype()` before validation.

`coerce` can be set at:
- **Column level:** `pa.Column(int, coerce=True)` — coerce only that column.
- **Schema level:** `pa.DataFrameSchema({...}, coerce=True)` — coerce all columns.

Coercion fails with a `SchemaError` if the cast would produce `NaN` (e.g., `"abc"` → `int`).


In [ ]:
# Simulate data arriving from a CSV — all columns are strings
df_raw = pd.DataFrame({
    "name":   ["Alice", "Bob", "Carol"],
    "age":    ["30", "25", "40"],        # <-- strings, not ints
    "salary": ["90000.0", "75000.0", "120000.0"],
})

print("Before validation dtypes:")
print(df_raw.dtypes)

# Schema with schema-level coerce=True
schema_coerce = pa.DataFrameSchema(
    {
        "name":   pa.Column(str),
        "age":    pa.Column(int),    # will be cast from string
        "salary": pa.Column(float),  # will be cast from string
    },
    coerce=True,   # applies to all columns
)

validated = schema_coerce.validate(df_raw)
print("\nAfter validation dtypes:")
print(validated.dtypes)
print("\nValidated DataFrame:")
print(validated)


### What just happened?

- **`coerce=True` at schema level saves you from manual `.astype()` chains** — declare the target type once.
- The validated DataFrame has proper `int64` and `float64` dtypes, not `object`.
- Coercion happens *before* checks run — so you can add `Check.greater_than(0)` on a column that arrives as strings.
- **Column-level `coerce` overrides schema-level** when both are set — useful for one column that must stay as string while the rest coerce.


## Step 4 · Triggering a SchemaError — reading `failure_cases`

When validation fails, Pandera raises `pa.errors.SchemaError`. The exception
carries a `failure_cases` attribute — a DataFrame with the **exact rows and
values** that violated the schema. This is your debugging starting point.

```python
except pa.errors.SchemaError as e:
    print(e.failure_cases)   # DataFrame: index + failing value
```

You can also pass `lazy=True` to `.validate()` to **collect all failures at once**
instead of stopping at the first one (raises `SchemaErrors`, plural).


In [ ]:
# Intentionally bad data: negative age, and a name that's actually an int
df_bad = pd.DataFrame({
    "name":   ["Alice", "Bob", "Carol"],
    "age":    [30, -5, 40],           # -5 is invalid (we'll add a check below)
    "salary": [90000.0, 75000.0, 120000.0],
})

schema_with_check = pa.DataFrameSchema({
    "name":   pa.Column(str),
    "age":    pa.Column(int, pa.Check.greater_than(0)),   # age must be positive
    "salary": pa.Column(float),
})

try:
    schema_with_check.validate(df_bad)
except pa.errors.SchemaError as e:
    print("=== SchemaError raised ===")
    print("Schema context:", e.schema)     # which schema/column failed
    print("\nFailure cases DataFrame:")
    print(e.failure_cases)                 # the offending rows
    print("\nFull error message:", str(e))


### What just happened?

- **`e.failure_cases`** gives you a DataFrame with an `index` column (row number) and `failure_case` column (the bad value) — far more useful than a generic `AssertionError`.
- The message shows the column name, the check that fired, and the offending value.
- **Tip:** In production pipelines, log `e.failure_cases.to_dict()` to your monitoring system — it's JSON-serializable.
- Use `schema.validate(df, lazy=True)` if you want all failures before aborting, not just the first.


## Step 5 · Schema-level vs Column-level coerce

Understanding where to put `coerce=True` matters when different columns have
different casting requirements.

| Setting | Scope | Best for |
|---|---|---|
| `pa.DataFrameSchema({...}, coerce=True)` | All columns | Uniform CSV ingestion |
| `pa.Column(int, coerce=True)` | One column only | Mixed sources |
| Column-level overrides schema-level | — | Fine-grained control |

Official docs: https://pandera.readthedocs.io/en/stable/dataframe_schemas.html#coercing-types


In [ ]:
# Mixed source: name stays object (already str), age is a string from CSV,
# salary is already float — only age needs coercion.

df_mixed = pd.DataFrame({
    "name":   ["Alice", "Bob"],
    "age":    ["30", "25"],       # string
    "salary": [90000.0, 75000.0], # already float
})

# Column-level coerce on just `age`
schema_partial_coerce = pa.DataFrameSchema({
    "name":   pa.Column(str),                  # no coerce needed
    "age":    pa.Column(int, coerce=True),      # only this column coerced
    "salary": pa.Column(float),                # no coerce needed
})

result = schema_partial_coerce.validate(df_mixed)
print("Dtypes after partial coercion:")
print(result.dtypes)
print("\nResult:")
print(result)

# Schema-level coerce=True also works here — but explicit is better
print("\nColumn-level coerce is more precise for mixed sources.")


### What just happened?

- **Column-level `coerce=True`** is surgical — only the column that needs casting gets touched.
- Schema-level `coerce=True` is a convenience shorthand when all columns come from a string source (CSV, JSON).
- **Coercion failures are schema errors:** if `"abc"` can't cast to `int`, Pandera raises `SchemaError` just like a dtype mismatch — no silent `NaN` injection.
- Pandera docs recommend column-level coerce for production schemas: it makes the contract explicit and catches unexpected type changes in upstream data.


In [ ]:
# Challenge: Build a schema for a user_events table with these requirements:
#   - user_id: int, no nulls
#   - event_type: str, no nulls
#   - duration_sec: float, nullable (event may be instantaneous)
#   - count: int — data arrives as strings, so coerce it
#
# Then validate this sample DataFrame and inspect failure_cases if it fails.

df_events = pd.DataFrame({
    "user_id":      [1, 2, 3],
    "event_type":   ["click", "view", "purchase"],
    "duration_sec": [0.5, None, 2.3],
    "count":        ["3", "1", "7"],     # strings — needs coercion
})

# Your schema here:
# schema_events = pa.DataFrameSchema({
#     ...
# })

# Your validation here:
# try:
#     result = schema_events.validate(df_events)
#     print(result)
# except pa.errors.SchemaError as e:
#     print(e.failure_cases)


---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| `DataFrameSchema` | Dict-style schema; `.validate(df)` returns the df or raises `SchemaError` |
| `pa.Column(dtype)` | Declares expected dtype; strict by default (no nulls, no coercion) |
| `nullable=True` | Opt-in to allow `NaN` — document your contract explicitly |
| `coerce=True` | Auto-cast to declared dtype; fails loudly if cast is impossible |
| `SchemaError.failure_cases` | DataFrame of offending rows — your debugging entry point |
| Schema-level vs Column-level coerce | Schema-level = all columns; Column-level = surgical |

> **Tip:** Always validate at the **boundary** — when data enters your pipeline (from CSV, API, DB). Validating too late means bad data has already propagated.

---
## What's next
**Day 2** → Built-in Checks (`Check.greater_than`, `Check.isin`, `Check.str_matches`) and custom lambda checks — add value-level constraints on top of your type schema.

Mark Day 1 complete in your [tracker](../index.html).
